---
title: "并查集"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [10]:
 #| code-fold: true
class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.count = n

    def find(self, i: int) -> int:
        """查找"""
        if self.parent[i] != i:
            self.parent[i] = self.find(self.parent[i])
        return self.parent[i]

    def union(self, i: int, j: int) -> bool:
        """合并"""
        root_i = self.find(i)
        root_j = self.find(j)

        if root_i != root_j:
            self.parent[root_i] = root_j
            self.count -= 1
            return True
        return False

    def is_connected(self, i: int, j: int) -> bool:
        """判断是不是同一个"""
        return self.find(i) == self.find(j)

## 📝 题目 547：省份数量

::: {.callout-caution}
### 📖 题目描述
有 `n` 个城市，其中一些彼此相连，另一些没有相连。如果城市 `a` 与城市 `b` 直接相连，且城市 `b` 与城市 `c` 直接相连，那么城市 `a` 与城市 `c` 间接相连。
**省份** 是一组直接或间接相连的城市，组内不含其他没有相连的城市。

给你一个 `n x n` 的矩阵 `isConnected` ，其中 `isConnected[i][j] = 1` 表示第 `i` 个城市和第 `j` 个城市直接相连，而 `isConnected[i][j] = 0` 表示二者不直接相连。
返回矩阵中 **省份** 的数量。

**输入输出示例**：

* **示例 1**：
    * **输入**：`isConnected = [[1,1,0],[1,1,0],[0,0,1]]`
    * **输出**：`2`
    * **解释**：城市 0 和 1 相连，形成一个省份。城市 2 孤立，形成第二个省份。

* **示例 2**：
    * **输入**：`isConnected = [[1,0,0],[0,1,0],[0,0,1]]`
    * **输出**：`3`
    * **解释**：大家各自为政，互不相连，一共 3 个省份。
:::

In [2]:
class Solution547:
    def findCircleNum(self, isConnected: list[list[int]]) -> int:
        n = len(isConnected)
        uf = UnionFind(n)

        for i in range(n):
            for j in range(i + 1, n):
                if isConnected[i][j] == 1:
                    uf.union(i, j)
        return uf.count

In [3]:
#| code-fold: true
def test_547(func):
    test_cases = [
        {
            "desc": "示例 1: 两个省份",
            "matrix": [[1,1,0],[1,1,0],[0,0,1]],
            "expected": 2
        },
        {
            "desc": "示例 2: 各自为政 (3个独立岛)",
            "matrix": [[1,0,0],[0,1,0],[0,0,1]],
            "expected": 3
        },
        {
            "desc": "全图大一统 (1个省份)",
            "matrix": [[1,1,1],[1,1,1],[1,1,1]],
            "expected": 1
        },
        {
            "desc": "单城市边界",
            "matrix": [[1]],
            "expected": 1
        },
        {
            "desc": "链式传递 (A-B-C-D)",
            "matrix": [
                [1,1,0,0],
                [1,1,1,0],
                [0,1,1,1],
                [0,0,1,1]
            ],
            "expected": 1
        },
        {
            "desc": "交错两个帮派",
            "matrix": [
                [1,0,1,0],
                [0,1,0,1],
                [1,0,1,0],
                [0,1,0,1]
            ],
            "expected": 2
        }
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["matrix"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<3} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_547(Solution547().findCircleNum)

ID  | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1   | ✅ PASS | 2    | 2    | 示例 1: 两个省份
2   | ✅ PASS | 3    | 3    | 示例 2: 各自为政 (3个独立岛)
3   | ✅ PASS | 1    | 1    | 全图大一统 (1个省份)
4   | ✅ PASS | 1    | 1    | 单城市边界
5   | ✅ PASS | 1    | 1    | 链式传递 (A-B-C-D)
6   | ✅ PASS | 2    | 2    | 交错两个帮派
-----------------------------------------------------------------
测试总结: 通过 6/6


::: {.callout-important}
### 💡 思路讲解 (从递归到纯粹的状态机)

这题如果用 DFS/BFS，需要维护 `visited` 数组，不停地递归去染色，代码结构极其臃肿。

**并查集视角的降维打击**：
图论中的“无向图连通分量”问题，就是并查集的绝对主场。
1. 一开始，我们假设这 $N$ 个城市是 $N$ 个独立的省份（孤岛）。
2. 我们去扫描关系矩阵。因为关系是对称的（你连着我，就等于我连着你），为了追求极致速度，只扫 $j > i$ 的右上三角区域。
3. 一旦发现 `isConnected[i][j] == 1`，说明 $i$ 和 $j$ 是连通的，直接呼叫 `union(i, j)` 进行合并。
4. `union` 内部会自动判断：如果他们俩原本不属于同一个省份，合并成功，并将全图的总省份数量 `count` 减 1。
5. 扫描结束后，`count` 的值就是答案！
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N^2)$。
  我们需要遍历矩阵的右上半部分，大约是 $N^2 / 2$ 次操作。
  每次执行 `union` 和 `find`，得益于“路径压缩”黑魔法，均摊时间复杂度为极其恐怖的 $O(\alpha(N))$（阿克曼函数的反函数，在宇宙范围内的数据规模下，它都小于 5，可以视为绝对的 $O(1)$ 常数时间）。所以总时间就是 $O(N^2)$。
* **空间复杂度**：$O(N)$。
  我们只开辟了一个长度为 $N$ 的 `parent` 数组，比起 DFS 需要系统调用栈和 `visited` 数组，极其轻量！
:::

## 📝 题目 684：冗余连接

::: {.callout-caution}
### 📖 题目描述
在本问题中，树是一个 **无向图**，其中任意两个顶点间有且只有一条路径。
输入的图原本是一棵树，但有人在其中添加了一条额外的边，使得图中出现了一个环。

给你一个长度为 `n` 的二维数组 `edges`，表示所有的边 `edges[i] = [ai, bi]`。
请你找出这条可以删去的边。如果有多个答案，则返回二维数组中 **最后出现** 的那条边。

**输入输出示例**：

* **示例 1**：
    * **输入**：`edges = [[1,2], [1,3], [2,3]]`
    * **输出**：`[2,3]`
    * **解释**：1-2, 1-3 已经连通了，[2,3] 进来后形成了环。

* **示例 2**：
    * **输入**：`edges = [[1,2], [2,3], [3,4], [1,4], [1,5]]`
    * **输出**：`[1,4]`
:::

In [4]:
class Solution684:
    def findRedundantConnection(self, edges: list[list[int]]) -> list[int]:
        n = len(edges)
        # 节点从 1 开始，所以开 n+1
        uf = UnionFind(n + 1)

        for u, v in edges:
            # 尝试合并。如果 union 返回 False，说明 u 和 v 已经连通了
            if not uf.union(u, v):
                return [u, v]

        return []

In [6]:
#| code-fold: true
def test_684(func):
    test_cases = [
        # 基础测试
        {"desc": "1. 基础三角环", "edges": [[1,2], [1,3], [2,3]], "expected": [2,3]},
        {"desc": "2. 经典五边形带尾巴", "edges": [[1,2], [2,3], [3,4], [1,4], [1,5]], "expected": [1,4]},

        # 链状与环状变体
        {"desc": "3. 完美大闭环 (头尾相连)", "edges": [[1,2], [2,3], [3,4], [4,5], [5,1]], "expected": [5,1]},
        {"desc": "4. P字型图 (长链中间成环)", "edges": [[1,2], [2,3], [3,4], [4,5], [2,4]], "expected": [2,4]},
        {"desc": "5. 星型放射图 (中心点发散后外围连线)", "edges": [[1,2], [1,3], [1,4], [1,5], [4,5]], "expected": [4,5]},

        # 乱序与多孤岛合并 (极其考验并查集的合并逻辑)
        {"desc": "6. 乱序三角环 (节点编号非递增)", "edges": [[1,3], [3,2], [1,2]], "expected": [1,2]},
        {"desc": "7. 多孤岛同时发育，最后汇聚成环", "edges": [[1,2], [3,4], [2,3], [4,5], [1,5]], "expected": [1,5]},
        {"desc": "8. 极其混乱的输入顺序", "edges": [[3,4], [1,2], [2,4], [3,5], [2,5]], "expected": [2,5]},

        # 大规模深层树成环
        {"desc": "9. 十节点深层二叉树，叶子节点相连", "edges": [
            [1,2], [1,3], [2,4], [2,5], [3,6],
            [3,7], [4,8], [4,9], [5,10], [8,10]
        ], "expected": [8,10]},

        # 核心干扰项
        {"desc": "10. 环早就形成了，但后面还有合法边 (按题意找出最后一条导致环的边)",
         "edges": [[2,1], [3,1], [4,2], [1,4], [5,3]],
         "expected": [1,4]} # 1-4 导致 1-2-4 成环，虽然 5-3 在后面，但 5-3 是安全的叶子节点
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<8} | {'实际':<8} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["edges"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<3} | {status:<6} | {str(tc['expected']):<8} | {str(actual):<8} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_684(Solution684().findRedundantConnection)

ID  | 结果     | 预期       | 实际       | 描述
-----------------------------------------------------------------
1   | ✅ PASS | [2, 3]   | [2, 3]   | 1. 基础三角环
2   | ✅ PASS | [1, 4]   | [1, 4]   | 2. 经典五边形带尾巴
3   | ✅ PASS | [5, 1]   | [5, 1]   | 3. 完美大闭环 (头尾相连)
4   | ✅ PASS | [2, 4]   | [2, 4]   | 4. P字型图 (长链中间成环)
5   | ✅ PASS | [4, 5]   | [4, 5]   | 5. 星型放射图 (中心点发散后外围连线)
6   | ✅ PASS | [1, 2]   | [1, 2]   | 6. 乱序三角环 (节点编号非递增)
7   | ✅ PASS | [1, 5]   | [1, 5]   | 7. 多孤岛同时发育，最后汇聚成环
8   | ✅ PASS | [2, 5]   | [2, 5]   | 8. 极其混乱的输入顺序
9   | ✅ PASS | [8, 10]  | [8, 10]  | 9. 十节点深层二叉树，叶子节点相连
10  | ✅ PASS | [1, 4]   | [1, 4]   | 10. 环早就形成了，但后面还有合法边 (按题意找出最后一条导致环的边)
-----------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解：第一个 union 失败的时刻



这道题是并查集 `union` 返回值的完美秀场。
1. 实例化 `UnionFind(n)`。注意：这题节点编号通常是从 1 开始的，所以实例化时可以开 `n + 1` 大小。
2. 遍历 `edges` 数组中的每一条边 `[u, v]`。
3. 直接调用 `if not uf.union(u, v):`。
4. **逻辑**：
   - 如果 `union` 成功，说明 $u$ 和 $v$ 之前不通，现在连上了，安全。
   - 如果 `union` 失败，说明 $u$ 和 $v$ 的根节点（教父）已经是同一个了。这条边就是那个“多余的、导致成环的”最后一根稻草！
5. 直接返回当前的 `[u, v]`。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N \alpha(N))$。其中 $\alpha$ 是阿克曼函数的反函数，接近 $O(N)$。由于我们只遍历一次边数组，速度极快。
* **空间复杂度**：$O(N)$。仅需存储 parent 数组。
:::

## 📝 题目 990：等式方程的可满足性

::: {.callout-caution}
### 📖 题目描述
给定一个由表示变量之间关系的字符串组成的数组 `equations`。每个字符串 `equations[i]` 的长度为 4，并采用两种不同的形式之一：`"a==b"` 或 `"a!=b"`。
在这里，`a` 和 `b` 是小写字母（代表变量）。

如果可以给变量名分配值，使所有给定的方程都满足，返回 `true`；否则返回 `false`。

**输入输出示例**：

* **示例 1**：
    * **输入**：`["a==b","b!=a"]`
    * **输出**：`false`
    * **解释**：既然 a==b，那 a 就不可能不等于 b。

* **示例 2**：
    * **输入**：`["b==a","a==b"]`
    * **输出**：`true`

* **示例 3**：
    * **输入**：`["a==b","b==c","a!=c"]`
    * **输出**：`false`
    * **解释**：a 和 b 是同伙，b 和 c 是同伙，推导出 a 和 c 也是同伙。但最后说 a!=c，矛盾！
:::

In [7]:
class Solution990:
    def equationsPossible(self, equations: list[str]) -> bool:
        # 变量只有 26 个小写字母
        uf = UnionFind(26)

        # 第一遍：处理所有相等关系
        for eq in equations:
            if eq[1] == '=':
                u = ord(eq[0]) - ord('a')
                v = ord(eq[3]) - ord('a')
                uf.union(u, v)

        # 第二遍：检查不等关系是否冲突
        for eq in equations:
            if eq[1] == '!':
                u = ord(eq[0]) - ord('a')
                v = ord(eq[3]) - ord('a')
                if uf.is_connected(u, v):
                    return False

        return True

In [9]:
#| code-fold: true
def test_990(func):
    test_cases = [
        {"desc": "1. 基础直接矛盾", "input": ["a==b", "b!=a"], "expected": False},
        {"desc": "2. 基础一致性", "input": ["a==b", "b==a"], "expected": True},
        {"desc": "3. 传递性矛盾 (A=B, B=C, A!=C)", "input": ["a==b", "b==c", "a!=c"], "expected": False},
        {"desc": "4. 独立集合不冲突", "input": ["a==b", "c==d", "a!=c"], "expected": True},
        {"desc": "5. 自指矛盾 (A!=A)", "input": ["a!=a"], "expected": False},
        {"desc": "6. 复杂长链传递", "input": ["a==b","b==c","c==d","d==e","e!=a"], "expected": False},
        {"desc": "7. 多路合并后的矛盾", "input": ["a==b","c==d","a==c","b!=d"], "expected": False},
        {"desc": "8. 乱序输入 (不等式在前)", "input": ["a!=b","b==c","c==a"], "expected": False},
        {"desc": "9. 全等关系", "input": ["a==b","b==c","c==d","d==e","a==e"], "expected": True},
        {"desc": "10. 多个独立矛盾源", "input": ["a==b","b!=a","c==d","d!=c"], "expected": False},
        {"desc": "11. 26个字母满载测试", "input": [chr(ord('a')+i) + "==" + chr(ord('a')+i+1) for i in range(25)] + ["a!=z"], "expected": False}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<8} | {'实际':<8} | {'描述'}")
    print("-" * 80)

    for i, tc in enumerate(test_cases):
        actual = func(tc["input"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<3} | {status:<6} | {str(tc['expected']):<8} | {str(actual):<8} | {tc['desc']}")

    print("-" * 80)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_990(Solution990().equationsPossible)

ID  | 结果     | 预期       | 实际       | 描述
--------------------------------------------------------------------------------
1   | ✅ PASS | False    | False    | 1. 基础直接矛盾
2   | ✅ PASS | True     | True     | 2. 基础一致性
3   | ✅ PASS | False    | False    | 3. 传递性矛盾 (A=B, B=C, A!=C)
4   | ✅ PASS | True     | True     | 4. 独立集合不冲突
5   | ✅ PASS | False    | False    | 5. 自指矛盾 (A!=A)
6   | ✅ PASS | False    | False    | 6. 复杂长链传递
7   | ✅ PASS | False    | False    | 7. 多路合并后的矛盾
8   | ✅ PASS | False    | False    | 8. 乱序输入 (不等式在前)
9   | ✅ PASS | True     | True     | 9. 全等关系
10  | ✅ PASS | False    | False    | 10. 多个独立矛盾源
11  | ✅ PASS | False    | False    | 11. 26个字母满载测试
--------------------------------------------------------------------------------
测试总结: 通过 11/11


::: {.callout-important}
### 💡 思路讲解：先建帮派，再查矛盾

1. **变量映射**：因为变量只有小写字母 'a'-'z'，我们直接实例化一个大小为 26 的兵工厂：`uf = UnionFind(26)`。字母转换数字：`ord(char) - ord('a')`。
2. **第一遍扫描（建立连通性）**：
   - 遍历所有 `==` 的方程。
   - 调用 `uf.union(var1, var2)`，把相等的变量合并到同一个集合。
3. **第二遍扫描（验证唯一性）**：
   - 遍历所有 `!=` 的方程。
   - 调用 `uf.is_connected(var1, var2)`。
   - **核心逻辑**：如果 `is_connected` 返回 `True`，说明这两个本该不相等的变量竟然在同一个帮派里！立刻判定 `False`。
4. **顺利过关**：如果所有 `!=` 方程都验证通过，返回 `True`。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N)$。$N$ 是方程的数量。我们扫描了两遍数组，并查集的操作近乎 $O(1)$。
* **空间复杂度**：$O(1)$。虽然我们开了并查集，但数组长度固定为 26，是常数级别的。
:::